## Definitions

$\text{Accuracy} =
\frac{\text{correct classifications}}{\text{total classifications}}
= \frac{TP+TN}{TP+TN+FP+FN} $

$\text{Recall (or TPR)} =
\frac{\text{correctly classified actual positives}}{\text{all actual positives}}
= \frac{TP}{TP+FN} $

$\text{FPR} =
\frac{\text{incorrectly classified actual negatives}}
{\text{all actual negatives}}
= \frac{FP}{FP+TN}$

$\text{Precision} =
\frac{\text{correctly classified actual positives}}
{\text{everything classified as positive}}
= \frac{TP}{TP+FP} $

## Decision Tree

In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
import os

data_path = os.getenv("DATA_PATH", "no_env_set")

seed = 1337
df = pd.read_csv(data_path)  # your file

target = "Problem_SKU"
# One-hot encode Storage_Size (drop size_4 as baseline)
size_dummies = pd.get_dummies(df['Storage_Size'], prefix='Size', drop_first=True)

# Encode Defect_In_Linked_Receive as 0/1
defect_linked_num = df['Defect_In_Linked_Receive'].astype(int)
# Use standardized/normalized numeric features + simple encodings for categoricals
numeric_features = [
    "Global_SKU_Defect_Rate_%_std",
    "ABS_Volume_Difference_std",
    "Aisle_Hold_%_std",
    "#_Pick_Events_std",
    "#_Pick_Events_In_Clique_std",
    "#_Picks_std",
    "#_Picks_In_Clique_std",
    "Time_In_Loc_std",
    "Current_Max_Volume_std",
]

feature_cols = numeric_features + list(size_dummies.columns) + ['Defect_In_Linked_Receive']

# Combine all properly encoded features
X = df[numeric_features].copy()
X = pd.concat([X, size_dummies, defect_linked_num], axis=1)

y = df[target]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=seed, stratify=y
)

from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import classification_report, roc_auc_score

dt = DecisionTreeClassifier(
    max_depth=None,        # let it grow; you can cap later
    min_samples_leaf=50,   # mild regularization
    random_state=42,
)

dt.fit(X_train, y_train)

y_pred_dt = dt.predict(X_test)
y_proba_dt = dt.predict_proba(X_test)[:, 1]

print("Decision Tree")
print(classification_report(y_test, y_pred_dt, digits=3))
print("ROC AUC:", roc_auc_score(y_test, y_proba_dt))

import numpy as np

fi_dt = pd.Series(dt.feature_importances_, index=feature_cols).sort_values(ascending=False)
print(fi_dt)

Decision Tree
              precision    recall  f1-score   support

       False      0.945     0.987     0.965     50954
        True      0.621     0.270     0.377      4046

    accuracy                          0.934     55000
   macro avg      0.783     0.629     0.671     55000
weighted avg      0.921     0.934     0.922     55000

ROC AUC: 0.8146674136661815
#_Picks_In_Clique_std           0.308417
Aisle_Hold_%_std                0.146590
Current_Max_Volume_std          0.126987
#_Picks_std                     0.112884
ABS_Volume_Difference_std       0.101996
#_Pick_Events_In_Clique_std     0.091083
Global_SKU_Defect_Rate_%_std    0.064365
#_Pick_Events_std               0.017400
Time_In_Loc_std                 0.013041
Defect_In_Linked_Receive        0.008358
Size_size_3                     0.004815
Size_size_2                     0.003028
Size_size_4                     0.001036
dtype: float64


## XG Boost

In [2]:
from xgboost import XGBClassifier

xgb = XGBClassifier(
    objective="binary:logistic",
    n_estimators=200,
    max_depth=4,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_lambda=1.0,
    n_jobs=-1,
    random_state=seed,
    eval_metric="logloss",
)

xgb.fit(X_train, y_train)

y_pred_xgb = xgb.predict(X_test)
y_proba_xgb = xgb.predict_proba(X_test)[:, 1]

print("XGBoost")
print(classification_report(y_test, y_pred_xgb, digits=3))
print("ROC AUC:", roc_auc_score(y_test, y_proba_xgb))

fi_xgb = pd.Series(xgb.feature_importances_, index=feature_cols).sort_values(ascending=False)
print(fi_xgb)

results = pd.DataFrame({
    "model": ["DecisionTree", "XGBoost"],
    "roc_auc": [roc_auc_score(y_test, y_proba_dt),
                roc_auc_score(y_test, y_proba_xgb)],
})
print(results)

XGBoost
              precision    recall  f1-score   support

       False      0.946     0.990     0.968     50954
        True      0.703     0.283     0.404      4046

    accuracy                          0.938     55000
   macro avg      0.824     0.637     0.686     55000
weighted avg      0.928     0.938     0.926     55000

ROC AUC: 0.8742971571520675
#_Picks_In_Clique_std           0.291325
Aisle_Hold_%_std                0.122333
Current_Max_Volume_std          0.108169
#_Picks_std                     0.079347
Defect_In_Linked_Receive        0.061832
ABS_Volume_Difference_std       0.059224
#_Pick_Events_In_Clique_std     0.055219
Size_size_2                     0.054648
Size_size_3                     0.051175
Size_size_4                     0.048084
Global_SKU_Defect_Rate_%_std    0.043279
#_Pick_Events_std               0.014170
Time_In_Loc_std                 0.011194
dtype: float32
          model   roc_auc
0  DecisionTree  0.814667
1       XGBoost  0.874297
